In [1]:
import pandas as pd
import numpy as np
import requests
import rasterio
import geopandas as gpd
import joblib
import math
import io
import glob
import os
from shapely.geometry import Point

print("Libraries loaded")

Libraries loaded


In [2]:
xgb_model = joblib.load("landslide_xgboost.pkl")

print("XGBoost model loaded successfully")

XGBoost model loaded successfully


In [3]:
dem_folder = r"C:\Users\workr\landslide Data\Slope data"

def get_elevation(latitude, longitude):

    lat_tile = math.floor(latitude)
    lon_tile = math.floor(longitude)

    lat_name = f"n{lat_tile:02d}" if lat_tile >= 0 else f"s{abs(lat_tile):02d}"
    lon_name = f"e{lon_tile:03d}" if lon_tile >= 0 else f"w{abs(lon_tile):03d}"

    filename = f"{lat_name}_{lon_name}_1arc_v3.tif"
    filepath = os.path.join(dem_folder, filename)

    if not os.path.exists(filepath):
        return None

    with rasterio.open(filepath) as src:

        value = list(
            src.sample([(longitude, latitude)])
        )[0][0]

        if src.nodata is not None and value == src.nodata:
            return None

        if value == -32767:
            return None

        return float(value)

In [4]:
def get_slope(latitude, longitude):

    lat_tile = math.floor(latitude)
    lon_tile = math.floor(longitude)

    lat_name = f"n{lat_tile:02d}" if lat_tile >= 0 else f"s{abs(lat_tile):02d}"
    lon_name = f"e{lon_tile:03d}" if lon_tile >= 0 else f"w{abs(lon_tile):03d}"

    filename = f"{lat_name}_{lon_name}_1arc_v3.tif"
    filepath = os.path.join(dem_folder, filename)

    if not os.path.exists(filepath):
        return None

    with rasterio.open(filepath) as src:

        row, col = src.index(
            longitude,
            latitude
        )

        if row < 1 or row >= src.height - 1:
            return None

        if col < 1 or col >= src.width - 1:
            return None

        window = rasterio.windows.Window(
            col - 1,
            row - 1,
            3,
            3
        )

        elevation = src.read(
            1,
            window=window
        ).astype(float)

        if src.nodata is not None:
            if (elevation == src.nodata).any():
                return None

        elevation[elevation == -32767] = np.nan

        if np.isnan(elevation).any():
            return None

        dx = (
            111320
            * math.cos(math.radians(latitude))
            * abs(src.res[0])
        )

        dy = 111320 * abs(src.res[1])

        dz_dx = (
            elevation[1, 2]
            - elevation[1, 0]
        ) / (2 * dx)

        dz_dy = (
            elevation[2, 1]
            - elevation[0, 1]
        ) / (2 * dy)

        slope = math.degrees(
            math.atan(
                math.sqrt(
                    dz_dx**2 +
                    dz_dy**2
                )
            )
        )

        return float(slope)

In [5]:
rainfall_folder = r"C:\Users\workr\landslide Data\rainfall data"

rainfall_files = glob.glob(
    os.path.join(
        rainfall_folder,
        "IMERG-Final.CLIM.200006-202305.V07B*.tif"
    )
)

if not rainfall_files:
    raise FileNotFoundError(
        "IMERG rainfall GeoTIFF not found"
    )

rainfall_file = rainfall_files[0]

print("Rainfall file loaded")

Rainfall file loaded


In [6]:
def get_rainfall(latitude, longitude):

    with rasterio.open(rainfall_file) as src:

        value = list(
            src.sample([(longitude, latitude)])
        )[0][0]

        if src.nodata is not None and value == src.nodata:
            return None

        return float(value)

In [7]:
def get_land_cover(latitude, longitude):

    lat_start = math.floor(latitude / 3) * 3
    lon_start = math.floor(longitude / 3) * 3

    lat_name = (
        f"N{lat_start:02d}"
        if lat_start >= 0
        else f"S{abs(lat_start):02d}"
    )

    lon_name = (
        f"E{lon_start:03d}"
        if lon_start >= 0
        else f"W{abs(lon_start):03d}"
    )

    filename = (
        f"ESA_WorldCover_10m_2021_v200_"
        f"{lat_name}{lon_name}_Map.tif"
    )

    url = (
        "https://esa-worldcover.s3.eu-central-1.amazonaws.com/"
        f"v200/2021/map/{filename}"
    )

    try:

        with rasterio.Env(
            AWS_NO_SIGN_REQUEST="YES"
        ):

            with rasterio.open(url) as src:

                value = list(
                    src.sample(
                        [(longitude, latitude)]
                    )
                )[0][0]

        return int(value)

    except Exception:

        return None

In [8]:
def get_clay(latitude, longitude):

    url = (
        "https://maps.isric.org/mapserv"
        "?map=/map/clay.map"
    )

    params = [
        ("SERVICE", "WCS"),
        ("VERSION", "2.0.1"),
        ("REQUEST", "GetCoverage"),
        ("COVERAGEID", "clay_0-5cm_mean"),
        ("FORMAT", "GEOTIFF_INT16"),
        (
            "SUBSETTINGCRS",
            "http://www.opengis.net/def/crs/EPSG/0/4326"
        ),
        (
            "OUTPUTCRS",
            "http://www.opengis.net/def/crs/EPSG/0/4326"
        ),
        (
            "SUBSET",
            f"X({longitude-0.002},{longitude+0.002})"
        ),
        (
            "SUBSET",
            f"Y({latitude-0.002},{latitude+0.002})"
        )
    ]

    response = requests.get(
        url,
        params=params,
        timeout=30
    )

    if response.status_code != 200:
        return None

    with rasterio.open(
        io.BytesIO(response.content)
    ) as src:

        value = list(
            src.sample(
                [(longitude, latitude)]
            )
        )[0][0]

    return float(value) / 10

In [9]:
river_file = r"C:\Users\workr\landslide Data\river data\HydroRIVERS\HydroRIVERS_v10_as_shp\HydroRIVERS_v10_as.shp"

rivers = gpd.read_file(
    river_file,
    engine="pyogrio"
)

print("River segments loaded:", len(rivers))

River segments loaded: 1428959


In [10]:
def get_river_distance(latitude, longitude):

    zone = int(
        (longitude + 180) / 6
    ) + 1

    epsg = 32600 + zone

    nearby_rivers = rivers.cx[
        longitude - 2:longitude + 2,
        latitude - 2:latitude + 2
    ].copy()

    if len(nearby_rivers) == 0:
        return None

    point = gpd.GeoDataFrame(
        geometry=[
            Point(
                longitude,
                latitude
            )
        ],
        crs="EPSG:4326"
    )

    point = point.to_crs(epsg)

    nearby_rivers = nearby_rivers.to_crs(
        epsg
    )

    distances = (
        nearby_rivers.geometry.distance(
            point.geometry.iloc[0]
        )
    )

    return float(
        distances.min() / 1000
    )

In [11]:
def predict_location(latitude, longitude):

    print("Getting environmental features...")

    elevation = get_elevation(
        latitude,
        longitude
    )

    slope = get_slope(
        latitude,
        longitude
    )

    rainfall = get_rainfall(
        latitude,
        longitude
    )

    land_cover = get_land_cover(
        latitude,
        longitude
    )

    clay = get_clay(
        latitude,
        longitude
    )

    river_distance = get_river_distance(
        latitude,
        longitude
    )

    values = [
        elevation,
        slope,
        rainfall,
        land_cover,
        clay,
        river_distance
    ]

    if any(v is None for v in values):

        print()
        print(
            "Prediction failed: "
            "one or more environmental features "
            "are unavailable."
        )

        return None

    features = pd.DataFrame([
        {
            "Elevation": elevation,
            "Slope": slope,
            "Annual_Rainfall": rainfall,
            "Land_Cover": land_cover,
            "Clay_Content": clay,
            "Distance_to_River_km": river_distance
        }
    ])

    probability = (
        xgb_model
        .predict_proba(features)[0][1]
        * 100
    )

    print()
    print("Latitude:", latitude)
    print("Longitude:", longitude)
    print()

    print(
        "Elevation:",
        round(elevation, 2),
        "m"
    )

    print(
        "Slope:",
        round(slope, 2),
        "degrees"
    )

    print(
        "Annual Rainfall:",
        round(rainfall, 2),
        "mm"
    )

    print(
        "Land Cover:",
        land_cover
    )

    print(
        "Clay Content:",
        round(clay, 2),
        "%"
    )

    print(
        "Distance to River:",
        round(river_distance, 3),
        "km"
    )

    print()

    print(
        "Landslide Susceptibility:",
        round(probability, 2),
        "%"
    )

    return probability

In [12]:
latitude = 25.284
longitude = 91.721

predict_location(
    latitude,
    longitude
)

Getting environmental features...

Latitude: 25.284
Longitude: 91.721

Elevation: 1436.0 m
Slope: 5.9 degrees
Annual Rainfall: 4041.06 mm
Land Cover: 50
Clay Content: 35.6 %
Distance to River: 0.618 km

Landslide Susceptibility: 98.81 %


np.float32(98.814644)